In [0]:
%sql

--
-- This notebook contains the key analytical queries that produce
-- the ATLAS conservation findings. All queries read from the
-- Gold star schema — validated in Notebook 4.
--
-- Run Notebook 4 Data Quality first to confirm all tables
-- are intact before running analysis.

-- Pre-flight check
SELECT layer, table_name, rows FROM (
  SELECT 'GOLD' AS layer, 'dim_species' AS table_name, count(*) AS rows FROM mcphersonsharyn_gold.dim_species
  UNION ALL SELECT 'GOLD', 'dim_date', count(*) FROM mcphersonsharyn_gold.dim_date
  UNION ALL SELECT 'GOLD', 'dim_environment', count(*) FROM mcphersonsharyn_gold.dim_environment
  UNION ALL SELECT 'GOLD', 'fact_species_observations', count(*) FROM mcphersonsharyn_gold.fact_species_observations
)
ORDER BY table_name;
-- Confirms Gold tables intact before analysis
-- All should show non-zero row counts

In [0]:
%sql
SELECT
  s.species_common,
  s.ocean_basin,
  d.enso_phase,
  count(DISTINCT f.animal_id) AS unique_animals,
  count(*) AS total_observations,
  round(avg(e.sea_surface_temperature), 2) AS avg_sst,
  round(avg(e.chlorophyll_concentration), 4) AS avg_cc,
  round(avg(e.degree_heating_week), 2) AS avg_dhw,
  round(avg(e.habitat_stress_index), 2) AS avg_stress
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s
  ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d
  ON f.date_id = d.date_id
JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
WHERE d.enso_phase IS NOT NULL
GROUP BY s.species_common, s.ocean_basin, d.enso_phase
ORDER BY s.species_common, avg_sst DESC;
-- KEY FINDING: Reef Manta Ray La Nina avg_sst should exceed El Nino
-- All other species show expected El Nino warmer pattern
-- Confirms finding is regional Indo-Pacific anomaly not global artefact

In [0]:
%sql

-- ANALYSIS 2 — LONG TERM WARMING TREND
-- Year on year average SST in each species habitat
-- Shows whether ocean is warming over the 30 year baseline
-- ENSO phase shown to separate climate signal from natural variation

SELECT
  s.species_common,
  f.year,
  d.enso_phase,
  count(DISTINCT f.animal_id) AS unique_animals,
  round(avg(e.sea_surface_temperature), 2) AS avg_sst,
  round(avg(e.habitat_stress_index), 2) AS avg_stress
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d ON f.date_id = d.date_id
JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
WHERE e.sea_surface_temperature IS NOT NULL
GROUP BY s.species_common, f.year, d.enso_phase
ORDER BY s.species_common, f.year, d.enso_phase;
-- Year on year SST per species with ENSO context
-- Rising trend across years confirms long-term warming
-- ENSO phase alongside separates El Nino warm years from baseline warming

In [0]:
%sql

-- ANALYSIS 3 — 30 YEAR SST BASELINE
-- Annual average SST across the full study region 1993 to 2026
-- Not filtered to species locations — shows full environmental trend
-- This is the baseline against which the 2026 El Nino is compared

SELECT
  e.year,
  d.enso_phase,
  round(avg(e.sea_surface_temperature), 2) AS avg_sst,
  round(min(e.sea_surface_temperature), 2) AS min_sst,
  round(max(e.sea_surface_temperature), 2) AS max_sst,
  count(DISTINCT e.h3_res4) AS h3_cells_measured
FROM mcphersonsharyn_gold.dim_environment e
JOIN mcphersonsharyn_gold.dim_date d
  ON e.year = d.year
  AND e.month = d.month
WHERE e.sea_surface_temperature IS NOT NULL
GROUP BY e.year, d.enso_phase
ORDER BY e.year, d.enso_phase;
-- Full study region SST trend 1993 to 2026
-- Shows whether the entire ocean is warming not just species habitat
-- enso_phase shown alongside year for context
-- h3_cells_measured shows spatial coverage per year

In [0]:
%sql

-- ANALYSIS 4 — CORAL BLEACHING EVENTS TIMELINE
-- Shows bleaching severity by year across the study region
-- Identifies the major bleaching events in the 41 year record

SELECT
  e.year,
  d.enso_phase,
  count(CASE WHEN e.bleaching_alert_area >= 1 THEN 1 END) AS watch_events,
  count(CASE WHEN e.bleaching_alert_area >= 2 THEN 1 END) AS warning_events,
  count(CASE WHEN e.bleaching_alert_area >= 3 THEN 1 END) AS alert_events,
  round(avg(CASE WHEN e.degree_heating_week > 0 THEN e.degree_heating_week END), 2) AS avg_dhw,
  round(max(e.degree_heating_week), 2) AS max_dhw
FROM mcphersonsharyn_gold.dim_environment e
JOIN mcphersonsharyn_gold.dim_date d
  ON e.year = d.year
  AND e.month = d.month
WHERE e.degree_heating_week IS NOT NULL
GROUP BY e.year, d.enso_phase
ORDER BY e.year, d.enso_phase;
-- Shows bleaching severity per year with ENSO context
-- 1998 El Nino — first global bleaching event
-- 2010 El Nino — second global bleaching event
-- 2016 El Nino — third global bleaching event — worst on record
-- 2024 El Nino — fourth global bleaching event — current

In [0]:
%sql

-- ANALYSIS 5 — FISHING PRESSURE IN SPECIES HABITAT
-- Shows fishing effort overlap with species tracking locations
-- Gulf of Mexico species face both climate stress and fishing pressure
-- This is the cumulative threat analysis
SELECT
  s.species_common,
  f.year,
  count(DISTINCT f.animal_id) AS unique_animals,
  count(*) AS tracking_observations,
  round(avg(e.sea_surface_temperature), 2) AS avg_sst,
  round(avg(e.habitat_stress_index), 2) AS avg_stress
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
WHERE s.species_common IN ('Whale Shark', 'Sperm Whale')
AND e.sea_surface_temperature IS NOT NULL
GROUP BY s.species_common, f.year
ORDER BY s.species_common, f.year;
-- Gulf of Mexico species — climate stress by year
-- Next step: join to Global Fishing Watch data
-- to show fishing hours in same H3 cells as species observations


In [0]:
%sql

-- ANALYSIS 6 — COMPLETE SPECIES SUMMARY
-- Final summary table combining all key metrics per species
-- This is the headline table for the ATLAS findings report

SELECT
  s.species_common,
  s.species_latin,
  s.iucn_status,
  s.ocean_basin,
  s.study_region,
  s.individual_count,
  s.total_observations,
  count(DISTINCT f.year) AS years_tracked,
  round(avg(e.sea_surface_temperature), 2) AS lifetime_avg_sst,
  round(max(e.sea_surface_temperature), 2) AS max_sst_experienced,
  round(avg(e.habitat_stress_index), 2) AS lifetime_avg_stress,
  count(CASE WHEN e.habitat_stress_index >= 2 THEN 1 END) AS high_stress_observations,
  round(count(CASE WHEN e.habitat_stress_index >= 2 THEN 1 END) * 100.0
    / count(*), 1) AS high_stress_pct,
  round(avg(e.degree_heating_week), 2) AS avg_dhw,
  round(max(e.degree_heating_week), 2) AS max_dhw_experienced
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
WHERE e.sea_surface_temperature IS NOT NULL
GROUP BY
  s.species_common, s.species_latin, s.iucn_status,
  s.ocean_basin, s.study_region,
  s.individual_count, s.total_observations
ORDER BY s.species_common;
-- Complete species summary with lifetime environmental exposure
-- lifetime_avg_sst shows average conditions each species experienced
-- max_sst_experienced shows worst thermal conditions recorded
-- high_stress_pct shows what proportion of time in degraded habitat
-- max_dhw_experienced shows worst bleaching exposure

In [0]:
%sql

-- ANALYSIS 7 — 2026 EL NINO CONTEXT
-- Compares 2026 conditions to historical baseline
-- Shows how unprecedented the current El Nino is
-- Context for the Earth Engine satellite imagery

SELECT
  e.year,
  d.enso_phase,
  round(avg(e.sea_surface_temperature), 2) AS avg_sst,
  round(avg(e.sea_surface_temperature) -
    avg(avg(e.sea_surface_temperature)) OVER (), 2) AS sst_anomaly_vs_mean,
  count(CASE WHEN e.bleaching_alert_area >= 2 THEN 1 END) AS bleaching_warning_cells,
  count(CASE WHEN e.sea_surface_temperature > 30 THEN 1 END) AS thermal_stress_cells,
  round(avg(e.habitat_stress_index), 2) AS avg_stress
FROM mcphersonsharyn_gold.dim_environment e
JOIN mcphersonsharyn_gold.dim_date d
  ON e.year = d.year
  AND e.month = d.month
WHERE e.sea_surface_temperature IS NOT NULL
AND e.year >= 2015
GROUP BY e.year, d.enso_phase
ORDER BY e.year, d.enso_phase;
-- Focuses on 2015 to 2026 — the period of accelerating warming
-- sst_anomaly_vs_mean shows how each year compares to the 34 year average
-- bleaching_warning_cells shows spatial extent of bleaching
-- thermal_stress_cells shows how much ocean is above 30C threshold
-- 2026 should show highest values in the record

In [0]:
%sql
-- What is the actual La Nina baseline vs current La Nina SST
-- in Reef Manta Ray habitat?
SELECT
  d.enso_phase,
  count(DISTINCT e.year) AS years_covered,
  round(avg(e.sea_surface_temperature), 2) AS avg_sst,
  round(min(e.sea_surface_temperature), 2) AS min_sst,
  round(max(e.sea_surface_temperature), 2) AS max_sst
FROM mcphersonsharyn_gold.dim_environment e
JOIN mcphersonsharyn_gold.dim_date d
  ON e.year = d.year
  AND e.month = d.month
JOIN mcphersonsharyn_gold.dim_location l
  ON e.h3_res4 = l.h3_res4
WHERE l.study_region = 'Raja Ampat Indonesia'
AND e.sea_surface_temperature IS NOT NULL
GROUP BY d.enso_phase
ORDER BY avg_sst DESC;
-- Shows actual SST averages in Raja Ampat by ENSO phase
-- across the full 34 year record
-- This gives us the real baseline numbers for the caption

In [0]:
%sql
SELECT
  CASE WHEN e.year BETWEEN 1993 AND 2005 THEN 'Early baseline 1993-2005'
       WHEN e.year BETWEEN 2006 AND 2015 THEN 'Mid period 2006-2015'
       WHEN e.year BETWEEN 2016 AND 2026 THEN 'Recent 2016-2026'
  END AS period,
  round(avg(e.sea_surface_temperature), 2) AS avg_sst
FROM mcphersonsharyn_gold.dim_environment e
JOIN mcphersonsharyn_gold.dim_location l ON e.h3_res4 = l.h3_res4
WHERE l.study_region = 'Raja Ampat Indonesia'
AND e.sea_surface_temperature IS NOT NULL
GROUP BY
  CASE WHEN e.year BETWEEN 1993 AND 2005 THEN 'Early baseline 1993-2005'
       WHEN e.year BETWEEN 2006 AND 2015 THEN 'Mid period 2006-2015'
       WHEN e.year BETWEEN 2016 AND 2026 THEN 'Recent 2016-2026'
  END
ORDER BY period;
-- Shows whether Raja Ampat SST is rising over the 34 year period
-- Compares early baseline to recent years
-- This gives us the warming trend number for the caption


In [0]:
%sql
SELECT
  f.animal_id,
  s.species_common,
  s.iucn_status,
  f.latitude,
  f.longitude,
  f.year,
  f.month,
  e.sea_surface_temperature,
  e.habitat_stress_index,
  e.degree_heating_week,
  e.bleaching_alert_category,
  d.enso_phase
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d ON f.date_id = d.date_id
LEFT JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
WHERE f.latitude IS NOT NULL
AND f.longitude IS NOT NULL
ORDER BY s.species_common, f.animal_id, f.year, f.month;
-- Export this as CSV for Kepler.gl
-- Contains all 69,776 tracking observations
-- With environmental context for each observation

In [0]:
%sql
SELECT
  f.animal_id,
  s.species_common,
  s.iucn_status,
  f.latitude,
  f.longitude,
  f.year,
  f.month,
  e.sea_surface_temperature,
  e.habitat_stress_index,
  e.degree_heating_week,
  e.bleaching_alert_category,
  d.enso_phase
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d ON f.date_id = d.date_id
LEFT JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
WHERE s.species_common = 'Reef Manta Ray'
AND f.latitude IS NOT NULL
AND f.longitude IS NOT NULL
ORDER BY f.animal_id, f.year, f.month;
-- Only 3,224 rows — well under the 10,000 limit
-- Download as CSV — ATLAS_Manta_Ray.csv

In [0]:
%sql
SELECT
  f.animal_id,
  s.species_common,
  s.iucn_status,
  f.latitude,
  f.longitude,
  f.year,
  f.month,
  e.sea_surface_temperature,
  e.habitat_stress_index,
  e.degree_heating_week,
  e.bleaching_alert_category,
  d.enso_phase
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d ON f.date_id = d.date_id
LEFT JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
WHERE s.species_common = 'Whale Shark'
AND f.latitude IS NOT NULL
AND f.longitude IS NOT NULL
ORDER BY f.animal_id, f.year, f.month;


In [0]:
%sql
SELECT f.animal_id, s.species_common, s.iucn_status,
  f.latitude, f.longitude, f.year, f.month,
  e.sea_surface_temperature, e.habitat_stress_index,
  e.degree_heating_week, e.bleaching_alert_category, d.enso_phase
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d ON f.date_id = d.date_id
LEFT JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4 AND f.year = e.year AND f.month = e.month
WHERE s.species_common = 'Sperm Whale'
AND f.latitude IS NOT NULL AND f.longitude IS NOT NULL
ORDER BY f.animal_id, f.year, f.month;

In [0]:
%sql
SELECT
  f.animal_id,
  s.species_common,
  s.iucn_status,
  f.latitude,
  f.longitude,
  f.year,
  f.month,
  e.sea_surface_temperature,
  e.habitat_stress_index,
  e.degree_heating_week,
  e.bleaching_alert_category,
  d.enso_phase
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d ON f.date_id = d.date_id
LEFT JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
WHERE s.species_common = 'Green Turtle'
AND f.latitude IS NOT NULL
AND f.longitude IS NOT NULL
AND f.year BETWEEN 1993 AND 2005
ORDER BY f.animal_id, f.year, f.month;

In [0]:
%sql
SELECT 
  MIN(f.year) as min_year,
  MAX(f.year) as max_year,
  COUNT(*) as total_rows
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
WHERE s.species_common = 'Green Turtle';

In [0]:
%sql
SELECT
  f.animal_id, s.species_common, s.iucn_status,
  f.latitude, f.longitude, f.year, f.month,
  e.sea_surface_temperature, e.habitat_stress_index,
  e.degree_heating_week, e.bleaching_alert_category, d.enso_phase
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d ON f.date_id = d.date_id
LEFT JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4 AND f.year = e.year AND f.month = e.month
WHERE s.species_common = 'Green Turtle'
AND f.latitude IS NOT NULL AND f.longitude IS NOT NULL
AND f.year BETWEEN 2012 AND 2014
ORDER BY f.animal_id, f.year, f.month;

In [0]:
%sql
SELECT
  f.animal_id, s.species_common, s.iucn_status,
  f.latitude, f.longitude, f.year, f.month,
  e.sea_surface_temperature, e.habitat_stress_index,
  e.degree_heating_week, e.bleaching_alert_category, d.enso_phase
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d ON f.date_id = d.date_id
LEFT JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4 AND f.year = e.year AND f.month = e.month
WHERE s.species_common = 'Green Turtle'
AND f.latitude IS NOT NULL AND f.longitude IS NOT NULL
AND f.year = 2012
ORDER BY f.animal_id, f.year, f.month;

In [0]:
import os
os.makedirs('/dbfs/FileStore', exist_ok=True)

df_turtle = spark.sql("""
SELECT
  f.animal_id,
  s.species_common,
  s.iucn_status,
  f.latitude,
  f.longitude,
  f.year,
  f.month,
  e.sea_surface_temperature,
  e.habitat_stress_index,
  e.degree_heating_week,
  e.bleaching_alert_category,
  d.enso_phase
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d ON f.date_id = d.date_id
LEFT JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
WHERE s.species_common = 'Green Turtle'
AND f.latitude IS NOT NULL
AND f.longitude IS NOT NULL
ORDER BY f.animal_id, f.year, f.month
""")

df_turtle.toPandas().to_csv('/dbfs/FileStore/ATLAS_Green_Turtle.csv', index=False)
print(f"Done: {df_turtle.count()} rows saved")

In [0]:
df_turtle = spark.sql("""
SELECT
  f.animal_id,
  s.species_common,
  s.iucn_status,
  f.latitude,
  f.longitude,
  f.year,
  f.month,
  e.sea_surface_temperature,
  e.habitat_stress_index,
  e.degree_heating_week,
  e.bleaching_alert_category,
  d.enso_phase
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d ON f.date_id = d.date_id
LEFT JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
WHERE s.species_common = 'Green Turtle'
AND f.latitude IS NOT NULL
AND f.longitude IS NOT NULL
ORDER BY f.animal_id, f.year, f.month
""")

df_turtle.toPandas().to_csv('/tmp/ATLAS_Green_Turtle.csv', index=False)
print(f"Done: {df_turtle.count()} rows")

In [0]:
import shutil
shutil.copy('/tmp/ATLAS_Green_Turtle.csv', '/Volumes/mcphersonsharyn_catalog/mcphersonsharyn_bronze/raw_files/ATLAS_Green_Turtle.csv')
print("Copied to Volume")

In [0]:
# Check what volumes exist
display(spark.sql("SHOW VOLUMES IN mcphersonsharyn_catalog.mcphersonsharyn_bronze"))

In [0]:
display(spark.sql("SHOW CATALOGS"))

In [0]:
display(spark.sql("SHOW VOLUMES IN workspace"))

In [0]:
display(spark.sql("SHOW SCHEMAS IN workspace"))

In [0]:
display(spark.sql("SHOW VOLUMES IN workspace.mcphersonsharyn_bronze"))

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.mcphersonsharyn_bronze.exports")

In [0]:
df_turtle.toPandas().to_csv('/Volumes/workspace/mcphersonsharyn_bronze/exports/ATLAS_Green_Turtle.csv', index=False)
print("Done")

In [0]:
%sql
SELECT year, SUM(bleaching_alert_area) as bleaching_warning_cells
FROM workspace.mcphersonsharyn_gold.dim_environment
GROUP BY year
ORDER BY year;